In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn import metrics, tree
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score,                              
                             f1_score,
                             confusion_matrix, ConfusionMatrixDisplay)

Create ouput directory for plots

In [ ]:
os.makedirs("output", exist_ok=True)

Define a random variable for reproducibility

In [ ]:
random_variable = 1234

Set the file path

In [ ]:
file_path = "https://raw.github.com/jan-source/ppa_erp_student/main/sessions/2_session/"

#### Use ";" as the separator:
data_a = pd.read_csv('churn_part_a.csv', delimiter = ';')

In [ ]:
data_a = pd.read_csv(f'{file_path}churn_part_a.csv')

In [ ]:
data_a.head()

In [ ]:
data_b = pd.read_csv(f'{file_path}churn_part_b.csv')

In [ ]:
data_b.head()

### Combine datasets
https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.merge.html

In [ ]:
data = pd.merge(data_a, data_b, how='inner', on='CustomerId')

In [ ]:
data.head()

In [ ]:
data.shape

In [ ]:
data_a.shape

In [ ]:
data_b.shape

### %-Anteil der Kunden die uns verlassen haben

In [ ]:
data["Exited"].sum()/data.shape[0]

### Determine the number of complete records

In [ ]:
data.dropna(axis=0).shape

### Remove rows where either Gender or Age, or both, are NaN

In [ ]:
data.dropna(axis=0, subset=['Gender', 'Age']).shape

In [ ]:
data.shape

In [ ]:
### Create a dataset with rows containing NaNs removed

In [ ]:
data_cleaned = data.dropna(axis=0)

In [ ]:
data_cleaned.shape

### Visual analysis

In [ ]:
plt.scatter(data_cleaned['CreditScore'], data_cleaned['Exited'])

In [ ]:
data_cleaned.groupby('Exited')['CreditScore'].mean()

In [ ]:
data_cleaned.groupby('Exited')['Age'].mean()

In [ ]:
data_cleaned.groupby('Exited')['Age'].min()

### Overview of churn by age group

In [ ]:
# Create age groups and analyze churn by age
# First, let's create age groups
data_cleaned.loc[:, 'Age_Group'] = pd.cut(
        data_cleaned['Age'], 
        bins=[17, 30, 40, 50, 60, 100], 
        labels=['18-30', '31-40', '41-50', '51-60', '60+']
        )

# Calculate churn rate for each age group
churn_by_age = data_cleaned.groupby('Age_Group')['Exited'].mean()

# Create the bar chart
plt.figure(figsize=(10, 6))
churn_by_age.plot(kind='bar', color='blue', alpha=0.7)
plt.title('Customer Churn Rate by Age Group', fontsize=14)
plt.xlabel('Age Group', fontsize=12)
plt.ylabel('Churn Rate', fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Add percentage labels on top of each bar
for i, value in enumerate(churn_by_age):
    plt.text(i, value + 0.01, f'{value:.1%}', ha='center', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# Create age groups and analyze churn by age with counts
data_cleaned.loc[:, 'Age_Group'] = pd.cut(
        data_cleaned['Age'], 
        bins=[17, 30, 40, 50, 60, 100], 
        labels=['18-30', '31-40', '41-50', '51-60', '60+']
        )

# Calculate churn rate and counts for each age group
age_group_stats = data_cleaned.groupby('Age_Group').agg(
    churn_rate=('Exited', 'mean'),
    count=('Exited', 'count')
)

# Create the bar chart
plt.figure(figsize=(10, 6))
age_group_stats['churn_rate'].plot(kind='bar', color='blue', alpha=0.7)
plt.title('Customer Churn Rate by Age Group', fontsize=14)
plt.xlabel('Age Group', fontsize=12)
plt.ylabel('Churn Rate', fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)


plt.ylim(0, 0.7)  # Sets y-axis to range from 0 to 0.5 (or 50%)

# Add percentage labels and counts on top of each bar
for i, (rate, count) in enumerate(zip(age_group_stats['churn_rate'], age_group_stats['count'])):
    plt.text(i, rate + 0.01, f'{rate:.1%}\n(n={count})', ha='center', fontsize=11)

plt.tight_layout()
plt.show()

### 1. Prediction model: logistic regression

### Selektiere Features (Variablen die für die Vorhersage genutzt werden)

In [ ]:
x = data_cleaned[['Age', 'CreditScore', 'Balance']]

In [ ]:
log_reg_simple = LogisticRegression(random_state=random_variable)

In [ ]:
log_reg_simple.fit(x, data_cleaned['Exited'])

In [ ]:
log_reg_simple.coef_

### Zeige Koeffizienten formatiert an

In [ ]:
coefficients_formated =  pd.concat(
    [
     pd.Series(['Age', 'CreditScore', 'Balance']),
     pd.DataFrame(np.transpose(log_reg_simple.coef_))
    ],
     axis=1
 )

In [ ]:
coefficients_formated

### Calculate the probability that a customer churns (Exited = 1)

In [ ]:
predictions_logit_regression = log_reg_simple.predict_proba(x)

### Plot Predictions vs. Age

In [ ]:
plt.scatter(data_cleaned['Age'], data_cleaned['Exited'], color='blue', label='Actual Data')
plt.scatter(data_cleaned['Age'], predictions_logit_regression[:, 1], color='lightblue', label='Predicted Probability')
plt.xlabel('Age')
plt.ylabel('Churn')
plt.title('Logistic Regression')
plt.axhline(0, color='black',linewidth=0.5)
plt.axvline(0, color='black',linewidth=0.5)
plt.legend()
plt.show()

Each customer receives a probability that they stay with us (Exited=0) and a probability that they churn (Exited=1)

### Predict whether a customer churns (1) or stays (0); each number represents a customer


In [ ]:
log_reg_simple.predict(x)

### Compute confusion matrix

In [ ]:
predictions = log_reg_simple.predict(x)

confusion_matrix(observed values, predicted values, ordering)

In [ ]:
cnf_matrix = metrics.confusion_matrix(
                    data_cleaned['Exited'], 
                    predictions, 
                    labels = [True, False]
                )

In [ ]:
cnf_matrix

In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix=cnf_matrix.T, 
                             display_labels=['churn', 'no churn'])

# Create a figure and plot
fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(ax=ax, cmap='Blues')
plt.xlabel('Actual Condition')  # This will now be on the x-axis
plt.ylabel('Predicted Condition')  # This will now be on the y-axis
plt.title('Confusion Matrix for Customer Churn')
plt.show()

In [ ]:
accuracy = (125 + 7674)/(125 + 1911 + 286 + 7674)

In [ ]:
accuracy

In [ ]:
metrics.accuracy_score(data_cleaned['Exited'], 
                    predictions)

In [ ]:
precision = 113/(113 + 181)

In [ ]:
precision

In [ ]:
metrics.precision_score(data_cleaned['Exited'], 
                    predictions)

In [ ]:
metrics.recall_score(data_cleaned['Exited'], 
                    predictions)


In [ ]:
metrics.f1_score(data_cleaned['Exited'], 
                    predictions)

### Test Model with more variables

In [ ]:
data_cleaned.head()

In [ ]:
x = data_cleaned[['Age', 'CreditScore', 'Balance', 'EstimatedSalary']]

In [ ]:
log_reg_simple = LogisticRegression(random_state=random_variable)

In [ ]:
log_reg_simple.fit(x, data_cleaned['Exited'])

In [ ]:
log_reg_simple.coef_

In [ ]:
predictions = log_reg_simple.predict(x)

In [ ]:
def evaluate_model(y_true, y_pred):
    cnf_matrix = confusion_matrix(y_true, y_pred, labels=[True, False])
    disp = ConfusionMatrixDisplay(confusion_matrix=cnf_matrix.T, 
                                 display_labels=['churn', 'no churn'])

    # Create a figure and plot
    fig, ax = plt.subplots(figsize=(8, 6))
    disp.plot(ax=ax, cmap='Blues')
    plt.xlabel('Actual Condition')  # This will now be on the x-axis
    plt.ylabel('Predicted Condition')  # This will now be on the y-axis
    plt.title('Confusion Matrix for Customer Churn')
    plt.show()

    # Add precision, recall, and F1-score
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    print(f'Accuracy: {accuracy:.2f}')
    print(f'Precision: {precision:.2f}')
    print(f'Recall: {recall:.2f}')
    print(f'F1-score: {f1:.2f}')

# Example usage:
# plot_confusion_matrix(cnf_matrix)

In [ ]:
evaluate_model(data_cleaned['Exited'], predictions)

## Dummy Encodings for Categorical Variables 

### Create a column that takes the value True for women and False for men

In [ ]:
data_cleaned['Gender_Boolean'] = data['Gender'] == 'Female'

In [ ]:
data_cleaned.head()

### Convert to numeric values

In [ ]:
data_cleaned['Gender_numeric'] = data_cleaned['Gender_Boolean'].astype(int)

In [ ]:
data_cleaned.head()

In [ ]:
data_cleaned['Geography'].unique()

### Generate dummy encoding for the Geography column

In [ ]:
geography_dummies = pd.get_dummies(data_cleaned['Geography'], prefix="g")

In [ ]:
geography_dummies

### Merge data_cleaned and geography_dummies (add columns to data_cleaned)

In [ ]:
data_full = pd.concat([data_cleaned, geography_dummies], axis=1)

In [ ]:
data_full

### New modell with more variables

In [ ]:
x_full = data_full[['Age', 'CreditScore', 'Balance', 'EstimatedSalary', 'Gender_numeric',
                 'g_Spain', 'g_Germany']]

In [ ]:
log_reg_simple = LogisticRegression(random_state=random_variable)

In [ ]:
log_reg_simple.fit(x_full, data_full['Exited'])

In [ ]:
predictions_full = log_reg_simple.predict(x_full)

In [ ]:
evaluate_model(data_full['Exited'], predictions_full)

In [ ]:
coefficients_formated =  pd.concat(
    [
     pd.Series(['Age', 'CreditScore', 'Balance', 'EstimatedSalary', 'Gender_numeric',
                 'g_Spain', 'g_Germany']),
     pd.DataFrame(np.transpose(log_reg_simple.coef_))
    ],
     axis=1
 )

In [ ]:
coefficients_formated

### Decision Tree

Decision tree with 3 levels (depth)

In [ ]:
clf_tree = tree.DecisionTreeClassifier(random_state=random_variable, max_depth=3)

In [ ]:
clf_tree.fit(x_full, data_full['Exited'])

In [ ]:
plt.figure(figsize=(15, 10))
tree.plot_tree(
        clf_tree, 
        feature_names=x_full.columns, 
        class_names=['on time', 'late'], 
        filled=True
        )
plt.tight_layout()
plt.savefig('output/decision_tree_visualization.png', dpi=300, bbox_inches='tight')
plt.show()

Decision tree with 20 levels (depth)

In [ ]:
clf_tree = tree.DecisionTreeClassifier(random_state=random_variable, max_depth=20)

### Fit the tree to the data (training)

In [ ]:
clf_tree.fit(x_full, data_full['Exited'])

### Generate predictions

In [ ]:
predictions_tree = clf_tree.predict(x_full)

In [ ]:
evaluate_model(data_full['Exited'], predictions_tree)

### Plot Decision Tree
### ! Next Cell takes very long

In [ ]:
plt.figure(figsize=(15, 10))
tree.plot_tree(
        clf_tree, 
        feature_names=x_full.columns, 
        class_names=['on time', 'late'], 
        filled=True
        )
plt.tight_layout()
plt.savefig('output/decision_tree_visualization.png', dpi=300, bbox_inches='tight')
plt.show()

- Very high accuracy, but the tree fits the training data too closely (overfitting)
- The tree will not perform as well on new data

### Random forest
Create multiple trees and use the average probability as the prediction

### Random forest for which we set that each leaf/group must contain at least 20 observations (parameter: min_samples_leaf)

In [ ]:
rf = RandomForestClassifier(random_state = random_variable, min_samples_leaf=20)

In [ ]:
rf.fit(x_full, data_full['Exited'])

In [ ]:
predictions_rf = rf.predict(x_full)

In [ ]:
evaluate_model(data_full['Exited'], predictions_rf)

## Gradient boosting

In [ ]:
gbm = GradientBoostingClassifier(random_state = random_variable, n_estimators=100)

In [ ]:
gbm.fit(x_full, data_full['Exited'])

In [ ]:
predictions_gbm = gbm.predict(x_full)

In [ ]:
evaluate_model(data_full['Exited'], predictions_gbm)

# Test / train split

In [ ]:
from sklearn.model_selection import train_test_split


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    x_full, 
    data_full['Exited'], 
    test_size=0.2, 
    random_state=random_variable,
    stratify=data_full['Exited']  # Maintain the same class distribution in train and test
)


In [ ]:
gbm.fit(X_train, y_train)

In [ ]:
predictions_gbm_test = gbm.predict(X_test)

In [ ]:
predictions_gbm_train = gbm.predict(X_train)


### Check Results in Test Set vs Train Set

In [ ]:
evaluate_model(y_train, predictions_gbm_train)

In [ ]:
evaluate_model(y_test, predictions_gbm_test)

## Grid search for Hyperparameter Tuning

In [ ]:
# Define parameter grid to search
param_grid = {
    'n_estimators': [50, 100, 200, 500],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 4, 5],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}


### Create base GBM model

In [ ]:

gbm_grid = GradientBoostingClassifier(random_state=random_variable)


### Setup GridSearchCV

In [ ]:
grid_search = GridSearchCV(
    estimator=gbm_grid,
    param_grid=param_grid,
    cv=3,
    n_jobs=-1,
    scoring='accuracy',
    verbose=2 # shows progress of the grid search
    )

### Fit the grid search

In [ ]:

grid_search.fit(x_full, data_full['Exited'])


### Get best parameters and best estimator

In [ ]:

print("Best parameters:", grid_search.best_params_)
best_gbm = grid_search.best_estimator_

### Make predictions with best model

In [ ]:

predictions_best_gbm = best_gbm.predict(x_full)


In [ ]:
evaluate_model(data_full['Exited'], predictions_best_gbm)